# Stromal reference projection

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)


<b><font size=5 color=pink >Step 1: load the published 2023 organoid dataset</font></b>


In [ ]:
load(file.path(project_root, "data", "reference", "organoids23", "Organoids23.RData"))
DimPlot(Organoids23, label = TRUE, group.by = "celltype")


In [ ]:
table(Organoids23@meta.data$celltype)


In [ ]:
library(dplyr)

Organoids23$compartment <- dplyr::case_when(
  Organoids23$celltype %in% c(
    "Endothelium",
    "Fibroblast",
    "MSC"
  ) ~ "Stromal",

  Organoids23$celltype %in% c(
    "Erythroid",
    "HSPC",
    "Megakaryocyte",
    "Monocyte",
    "Myeloid Progenitor"
  ) ~ "Haematopoietic",

  TRUE ~ NA_character_
)

table(Organoids23$compartment, useNA = "ifany")

Organoids23_Stromal <- subset(
  Organoids23,
  subset = compartment == "Stromal"
)

#Organoids23_Haematopoietic <- subset(
  #Organoids23,
  #subset = compartment == "Haematopoietic"
#)

table(Organoids23_Stromal$celltype)
#table(Organoids23_Haematopoietic$celltype)


In [ ]:
table(Organoids23_Stromal@meta.data$orig.ident)


In [ ]:
table(Organoids23_Stromal@meta.data$data_set)


In [ ]:
table(Organoids23_Stromal@meta.data$sampleID)


In [ ]:
Organoids23_Stromal <- NormalizeData(Organoids23_Stromal)
Organoids23_Stromal <- FindVariableFeatures(Organoids23_Stromal, selection.method = "vst", nfeatures = 2000)

all.genes <- rownames(Organoids23_Stromal)
Organoids23_Stromal <- ScaleData(Organoids23_Stromal, features = all.genes)

Organoids23_Stromal <- RunPCA(Organoids23_Stromal, features = VariableFeatures(object = Organoids23_Stromal))

ElbowPlot(Organoids23_Stromal, ndims = 30)


In [ ]:
table(Organoids23_Stromal@meta.data$celltype)


In [ ]:
table(Organoids23_Stromal@meta.data$annotations)


In [ ]:
Organoids23_Stromal <- FindNeighbors(Organoids23_Stromal, dims = 1:10)
Organoids23_Stromal <- FindClusters(Organoids23_Stromal, resolution = 3)

Organoids23_Stromal <- RunUMAP(Organoids23_Stromal, dims = 1:10, return.model = TRUE)
Organoids23_Stromal <- RunTSNE(Organoids23_Stromal, dims = 1:10, return.model = TRUE)

p1 <- DimPlot(Organoids23_Stromal, reduction = "tsne", group.by = "celltype", label = TRUE) + ggtitle("t-SNE")
p2 <- DimPlot(Organoids23_Stromal, reduction = "umap", group.by = "celltype", label = TRUE) + ggtitle("UMAP")

p1/p2


In [ ]:
saveRDS(Organoids23_Stromal, file = file.path(project_root, "data", "processed", "Organoids23_Stromal.rds"))


In [ ]:
Organoids23_Stromal <- readRDS(file.path(project_root, "data", "processed", "Organoids23_Stromal.rds"))


<b><font size=5 color=pink >Step 2: load BMO stromal cells</font></b>


In [ ]:
### ============================================
### ============================================
Stromal <- readRDS(file.path(project_root, "data", "processed", "Stromal_clean_2_celltype.rds"))

Stromal@meta.data$group <- ""
Stromal@meta.data$group[Stromal@meta.data$orig.ident %in% c("Dynamic.25d.1", "Dynamic.25d.2", "Dynamic.25d.3")] <- "Dynamic.25d"
Stromal@meta.data$group[Stromal@meta.data$orig.ident %in% c("Dynamic.31d")] <- "Dynamic.31d"
Stromal@meta.data$group[Stromal@meta.data$orig.ident %in% c("Static.25d.1", "Static.25d.2", "Static.25d.3")] <- "Static.25d"
table(Stromal@meta.data$group)

Stromal@meta.data$celltype <- ""
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("41","28","2","12","13","40")] <- "Osteoblasts"
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("15","16","22","23")] <- "Osteoblasts pre."
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("31","33")] <- "Cycling Osteoblasts pre."
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("7","24","34")] <- "Osteochondral pre."
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("32","38")] <- "Chondrocytes"
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("1","3","5","6","8","26","29","35","11","17","19","0","4","10","14","20","30")] <- "MSCs"
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("9","37")] <- "Pericytes"
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("21")] <- "Adipocytes pre."
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("18","27","39")] <- "Proliferating MSCs"
Stromal@meta.data$celltype[Stromal@meta.data$seurat_clusters %in% c("25","36")] <- "Endothelial Cells"
table(Stromal@meta.data$celltype)

lineage_order <- c("MSCs","Proliferating MSCs","Osteochondral pre.","Chondrocytes",
                   "Osteoblasts pre.","Cycling Osteoblasts pre.","Osteoblasts",
                   "Adipocytes pre.","Pericytes","Endothelial Cells")
Stromal$celltype <- factor(Stromal$celltype, levels = lineage_order)

custom_cols_stro <- c(
  "MSCs" = "#D9E3E4", "Proliferating MSCs" = "#E39A94",
  "Osteochondral pre." = "#B3CDE3", "Chondrocytes" = "#80B1D3",
  "Osteoblasts pre." = "#CCEBC5", "Cycling Osteoblasts pre." = "#B2DF8A",
  "Osteoblasts" = "#33A02C", "Adipocytes pre." = "#FDB462",
  "Pericytes" = "#F2D7D0", "Endothelial Cells" = "#BC80BD"
)

DimPlot(Stromal, reduction = "tsne", group.by = "celltype",
        label = TRUE, repel = TRUE, cols = custom_cols_stro) +
  theme(aspect.ratio = 1)


<b><font size=5 color=pink >Step 3: load adult bone marrow</font></b>


In [ ]:
Adult_BM_Stromal <- readRDS(file.path(project_root, "data", "processed", "Adult_BM_Stromal.rds"))


In [ ]:
#Adult_BM_Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "Adult_BM_Haematopoietic.rds"))


In [ ]:
Adult_BM_Stromal[["umap"]]@misc$model


In [ ]:
table(Adult_BM_Stromal@meta.data$cluster_anno_l2)


In [ ]:
DimPlot(Adult_BM_Stromal, reduction = "umap", group.by = "cluster_anno_l2", label = TRUE)


In [ ]:
table(Adult_BM_Stromal@meta.data$cluster_anno_l2)


In [ ]:
library(Seurat)
library(ggplot2)

Adult_BM_Stromal$cluster_anno_l2 <- factor(
  Adult_BM_Stromal$cluster_anno_l2,
  levels = c(
    "Adipo-MSC",
    "AEC",
    "APOD+ MSC",
    "Fibro-MSC",
    "Osteo-MSC",
    "Osteoblast",
    "RNAlo MSC",
    "SEC",
    "THY1+ MSC",
    "VSMC"
  )
)

stromal_cols <- c(
  "Adipo-MSC"  = "#4DBBD5",
  "AEC"        = "#E64B35",
  "APOD+ MSC"  = "#00A087",
  "Fibro-MSC"  = "#3C5488",
  "Osteo-MSC"  = "#F39B7F",
  "Osteoblast" = "#8491B4",
  "RNAlo MSC"  = "#7E6148",
  "SEC"        = "#DC0000",
  "THY1+ MSC"  = "#91D1C2",
  "VSMC"       = "#B09C85"
)

p_adult_stromal <- DimPlot(
  Adult_BM_Stromal,
  reduction = "umap",
  group.by = "cluster_anno_l2",
  label = TRUE,
  repel = TRUE,
  label.size = 4,
  cols = stromal_cols
) +
  theme_classic() +
  labs(
    title = "Adult BM stromal cell types",
    color = "Cell type"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 15, color = "black"),
    axis.title = element_text(color = "black"),
    axis.text = element_text(color = "black"),
    legend.title = element_text(color = "black"),
    legend.text = element_text(color = "black")
  )

p_adult_stromal


In [ ]:
p_adult_stromal <- DimPlot(
  Adult_BM_Stromal,
  reduction = "umap",
  group.by = "cluster_anno_l2",
  label = TRUE,
  repel = TRUE,
  label.size = 4,
  cols = stromal_cols
) +
  NoAxes() +
  labs(
    title = "Adult BM stromal cell types",
    color = "Cell type"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 10, color = "black"),
    legend.title = element_text(color = "black"),
    legend.text = element_text(color = "black")
  )

p_adult_stromal


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "1_Adult_BM_Stromal_celltype_UMAP.pdf"),
  plot = p_adult_stromal,
  width = 4.5,
  height = 4,
  units = "in"
)


<b><font size=5 color=pink >Step 4: load fetal bone marrow</font></b>


In [ ]:
FBM_Stromal <- readRDS(file.path(project_root, "data", "processed", "FBM_Stromal.rds"))


In [ ]:
#FBM_Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "FBM_Haematopoietic.rds"))


In [ ]:
FBM_Stromal[["umap"]]@misc$model


In [ ]:
table(FBM_Stromal@meta.data$cell.labels)


In [ ]:
DimPlot(FBM_Stromal, reduction = "umap", group.by = "cell.labels", label = TRUE) + NoLegend()


In [ ]:
table(FBM_Stromal@meta.data$cell.labels)


In [ ]:
library(Seurat)
library(ggplot2)

FBM_Stromal$cell.labels <- factor(
  FBM_Stromal$cell.labels,
  levels = c(
    "adipo-CAR",
    "arteriolar fibroblast",
    "chondrocyte",
    "early osteoblast",
    "endosteal fibroblast",
    "erythroid macrophage",
    "immature EC",
    "monocytoid macrophage",
    "muscle",
    "muscle stem cell",
    "myofibroblast",
    "osteoblast",
    "osteoblast precursor",
    "osteochondral precursor",
    "osteoclast",
    "proliferating EC",
    "schwann cells",
    "sinusoidal EC",
    "stromal macrophage",
    "tip EC"
  )
)

fbm_cols <- c(
  "adipo-CAR"                = "#E64B35",
  "arteriolar fibroblast"    = "#4DBBD5",
  "chondrocyte"              = "#00A087",
  "early osteoblast"         = "#3C5488",
  "endosteal fibroblast"     = "#F39B7F",
  "erythroid macrophage"     = "#8491B4",
  "immature EC"              = "#91D1C2",
  "monocytoid macrophage"    = "#DC0000",
  "muscle"                   = "#7E6148",
  "muscle stem cell"         = "#B09C85",
  "myofibroblast"            = "#1F78B4",
  "osteoblast"               = "#33A02C",
  "osteoblast precursor"     = "#FB9A99",
  "osteochondral precursor"  = "#E31A1C",
  "osteoclast"               = "#FDBF6F",
  "proliferating EC"         = "#FF7F00",
  "schwann cells"            = "#CAB2D6",
  "sinusoidal EC"            = "#6A3D9A",
  "stromal macrophage"       = "#B15928",
  "tip EC"                   = "#A6CEE3"
)

p_fbm_stromal <- DimPlot(
  FBM_Stromal,
  reduction = "umap",
  group.by = "cell.labels",
  label = TRUE,
  repel = TRUE,
  label.size = 3.5,
  cols = fbm_cols
) +
  NoAxes() +
  labs(
    title = "FBM stromal cell types",
    color = "Cell type"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    legend.title = element_text(color = "black"),
    legend.text = element_text(color = "black")
  )

p_fbm_stromal


In [ ]:
p_fbm_stromal + NoLegend()


In [ ]:
# Figures are written explicitly to results/figures; no working-directory change is required.


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "2_FBM_Stromal_celltype_UMAP.pdf"),
  plot = p_fbm_stromal,
  width = 6,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 5: map DBMOs to adult bone marrow</font></b>


In [ ]:
anchors.DBMOs <- FindTransferAnchors(
  reference = Adult_BM_Stromal,
  query = Stromal,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Stromal <- MapQuery(
  anchorset = anchors.DBMOs,
  reference = Adult_BM_Stromal,
  query = Stromal,
  refdata = list(ABM = "cluster_anno_l2"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Stromal@meta.data$predicted.ABM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(Adult_BM_Stromal@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Stromal@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Stromal@meta.data$predicted.ABM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "Projection onto ABM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "3_Projection_onto_ABM_UMAP.pdf"),
  plot = p_beautiful,
  width = 5,
  height = 4,
  units = "in"
)


<b><font size=5 color=pink >Step 5: map published organoids to adult bone marrow</font></b>


In [ ]:
anchors.23BMO <- FindTransferAnchors(
  reference = Adult_BM_Stromal,
  query = Organoids23_Stromal,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Organoids23_Stromal <- MapQuery(
  anchorset = anchors.23BMO,
  reference = Adult_BM_Stromal,
  query = Organoids23_Stromal,
  refdata = list(ABM = "cluster_anno_l2"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Organoids23_Stromal@meta.data$predicted.ABM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(Adult_BM_Stromal@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Organoids23_Stromal@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Organoids23_Stromal@meta.data$predicted.ABM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "mBMO Stromal Projection onto ABM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "4_mBMO_Stromal_Projection_onto_ABM_UMAP.pdf"),
  plot = p_beautiful,
  width = 5,
  height = 4,
  units = "in"
)


<b><font size=5 color=pink >Step 6: map DBMOs to fetal bone marrow</font></b>


In [ ]:
anchors.FBM.DBMOs <- FindTransferAnchors(
  reference = FBM_Stromal,
  query = Stromal,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Stromal <- MapQuery(
  anchorset = anchors.FBM.DBMOs,
  reference = FBM_Stromal,
  query = Stromal,
  refdata = list(FBM = "cell.labels"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Stromal@meta.data$predicted.FBM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(FBM_Stromal@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Stromal@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Stromal@meta.data$predicted.FBM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "DBMOs Stromal Projection onto FBM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "5_DBMOs_Stromal_Projection_onto_FBM_UMAP.pdf"),
  plot = p_beautiful,
  width = 5,
  height = 4,
  units = "in"
)


<b><font size=5 color=pink >Step 6: map published organoids to fetal bone marrow</font></b>


In [ ]:
anchors.FBM.BMO23 <- FindTransferAnchors(
  reference = FBM_Stromal,
  query = Organoids23_Stromal,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Organoids23_Stromal <- MapQuery(
  anchorset = anchors.FBM.BMO23,
  reference = FBM_Stromal,
  query = Organoids23_Stromal,
  refdata = list(FBM = "cell.labels"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Organoids23_Stromal@meta.data$predicted.FBM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(FBM_Stromal@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Organoids23_Stromal@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Organoids23_Stromal@meta.data$predicted.FBM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "mBMO Projection onto ABM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "6_mBMO_Stromal_Projection_onto_FBM_UMAP.pdf"),
  plot = p_beautiful,
  width = 5,
  height = 4,
  units = "in"
)


<b><font size=5 color=pink >Step 7: reproduce the final comparison figure</font></b>


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

# ============================================================
# ============================================================

score_df_Organoids23 <- Organoids23_Stromal@meta.data %>%
  dplyr::select(predicted.ABM.score, predicted.FBM.score) %>%
  dplyr::rename(
    ABM = predicted.ABM.score,
    FBM = predicted.FBM.score
  ) %>%
  tidyr::pivot_longer(
    cols = c(ABM, FBM),
    names_to = "Reference",
    values_to = "Score"
  ) %>%
  dplyr::mutate(
    Dataset = "BMO-2023"
  )

# ============================================================
# ============================================================

score_df_DBMOs <- Stromal@meta.data %>%
  dplyr::select(predicted.ABM.score, predicted.FBM.score) %>%
  dplyr::rename(
    ABM = predicted.ABM.score,
    FBM = predicted.FBM.score
  ) %>%
  tidyr::pivot_longer(
    cols = c(ABM, FBM),
    names_to = "Reference",
    values_to = "Score"
  ) %>%
  dplyr::mutate(
    Dataset = "DBMOs"
  )

# ============================================================
# ============================================================

plot_df <- dplyr::bind_rows(
  score_df_Organoids23,
  score_df_DBMOs
)

table(plot_df$Dataset, plot_df$Reference)
summary(plot_df$Score)


In [ ]:
# ============================================================
# ============================================================

p_score_density <- ggplot(
  plot_df,
  aes(x = Score, fill = Reference, color = Reference)
) +
  geom_density(
    alpha = 0.35,
    linewidth = 1,
    adjust = 1.5
  ) +
  facet_wrap(~ Dataset, nrow = 1) +
  coord_cartesian(xlim = c(0.1, 1.0)) +
  scale_fill_manual(
    values = c(
      "ABM" = "#d93025",
      "FBM" = "#1a73e8"
    )
  ) +
  scale_color_manual(
    values = c(
      "ABM" = "#d93025",
      "FBM" = "#1a73e8"
    )
  ) +
  theme_bw() +
  labs(
    title = "Projection mapping score to ABM and FBM",
    x = "Projection mapping score",
    y = "Cell density",
    fill = "Reference",
    color = "Reference"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 14,
      color = "black"
    ),
    strip.text = element_text(
      face = "bold",
      size = 12,
      color = "black"
    ),
    axis.text = element_text(color = "black"),
    axis.title = element_text(color = "black"),
    legend.position = "top",
    legend.title = element_text(color = "black"),
    legend.text = element_text(color = "black"),
    panel.grid.major = element_line(color = "#f0f0f0"),
    panel.grid.minor = element_blank()
  )

p_score_density


In [ ]:
library(ggplot2)
library(dplyr)

# ============================================================
# ============================================================

median_df <- plot_df %>%
  dplyr::group_by(Dataset, Reference) %>%
  dplyr::summarise(
    median_score = median(Score, na.rm = TRUE),
    .groups = "drop"
  )

p_score_density <- ggplot(
  plot_df,
  aes(x = Score, fill = Reference, color = Reference)
) +
  geom_density(
    alpha = 0.28,
    linewidth = 1.1,
    adjust = 1.4
  ) +

  geom_vline(
    data = median_df,
    aes(xintercept = median_score, color = Reference),
    linetype = "dashed",
    linewidth = 0.7,
    show.legend = FALSE
  ) +

  facet_wrap(~ Dataset, nrow = 1) +

  scale_x_continuous(
    limits = c(0.1, 1.0),
    breaks = seq(0.1, 1.0, by = 0.1),
    expand = c(0.01, 0.01)
  ) +

  scale_fill_manual(
    values = c(
      "ABM" = "#D73027",
      "FBM" = "#4575B4"
    )
  ) +
  scale_color_manual(
    values = c(
      "ABM" = "#D73027",
      "FBM" = "#4575B4"
    )
  ) +

  labs(
    title = "Projection mapping score to ABM and FBM",
    x = "Projection mapping score",
    y = "Cell density",
    fill = NULL,
    color = NULL
  ) +

  theme_classic(base_size = 13) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    strip.background = element_rect(
      fill = "#F2F2F2",
      color = NA
    ),
    strip.text = element_text(
      face = "bold",
      size = 12,
      color = "black"
    ),
    axis.title = element_text(
      size = 12,
      color = "black"
    ),
    axis.text = element_text(
      size = 10,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.5
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "top",
    legend.text = element_text(
      size = 11,
      color = "black"
    ),
    panel.spacing = unit(1.2, "lines")
  )

p_score_density


In [ ]:
outdir <- file.path(project_root, "data", "processed", "Figure3")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

ggsave(
  filename = file.path(outdir, "7_Stromal_projection_score_density_BMO2023_DBMOs_optimized.pdf"),
  plot = p_score_density,
  width = 6,
  height = 4.2,
  units = "in",
  device = "pdf"
)
